# DIAMOND Research Blueprint

This notebook is a downstream research blueprint for ColliderLake. It starts from the existing `muon_db` lakehouse and sketches the layers that turn curated physics tables into a research-grade collider data platform.

DIAMOND stands for:

- Detector and data-quality certification
- Inference-ready physics features
- Analysis marts and event selections
- Multi-object relations and graph structures
- Observability, validation, and lineage
- Novelty and anomaly candidate stores
- Dissemination-ready datasets and reports

In [ ]:
from pathlib import Path
import sys

WORKSPACE = Path.cwd()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent
sys.path.insert(0, str(WORKSPACE))

import pandas as pd

from src.access.muon_db import connect_with_tables
from src.research.registry import research_layer_rows

connection, tables = connect_with_tables(WORKSPACE / "data" / "muon_db")
pd.DataFrame(research_layer_rows())

## Current Lakehouse State

The current implementation gives us a real base to build on: bronze raw projections, silver cleaned physics objects, and gold curated tables.

In [ ]:
rows = []
for table in tables:
    rows.append({
        "layer": table.layer,
        "table": table.name,
        "rows": connection.execute(f"SELECT count(*) FROM {table.name}").fetchone()[0],
    })
pd.DataFrame(rows)

## Research Questions To Explore

Start with physics-grounded questions before adding advanced models:

1. What is the event yield after each quality and trigger cut?
2. Does the dimuon table reproduce the expected Z resonance structure?
3. Which high-ST events are driven by jet activity, MET, or muon pT?
4. What event regions are sparse in the current dataset?
5. Which object-pair topologies dominate the selected events?
6. Which candidate events remain interesting after certification and validation?

## Candidate Exploration: High ST Events

This is a deterministic anomaly-candidate view. It is not ML; it is a transparent rule-based starting point.

In [ ]:
connection.execute("""
SELECT
    event_id,
    n_muons,
    n_jets,
    leading_muon_pt,
    leading_jet_pt,
    MET_pt,
    HT,
    ST
FROM event_summary
ORDER BY ST DESC
LIMIT 25
""").fetchdf()

## Candidate Exploration: Z-Like Dimuon Pairs

The `dimuon` table contains opposite-sign cleaned muon pairs. This query finds pairs closest to the nominal Z mass.

In [ ]:
connection.execute("""
SELECT
    event_id,
    muon1_pt,
    muon2_pt,
    invariant_mass,
    delta_r,
    abs(invariant_mass - 91.1876) AS z_distance
FROM dimuon
WHERE invariant_mass BETWEEN 70 AND 110
ORDER BY z_distance
LIMIT 25
""").fetchdf()

## Sparse Region Sketch

Sparse regions are useful for deterministic anomaly-candidate discovery. This is a first-pass event-region binning using muon count, jet count, MET, and ST.

In [ ]:
connection.execute("""
SELECT
    n_muons,
    n_jets,
    floor(MET_pt / 25) * 25 AS met_bin_low,
    floor(ST / 100) * 100 AS st_bin_low,
    count(*) AS events
FROM event_summary
GROUP BY n_muons, n_jets, met_bin_low, st_bin_low
ORDER BY events ASC, st_bin_low DESC
LIMIT 50
""").fetchdf()

## Next Implementation Target

The strongest next build is the validation and cutflow layer. It should produce explicit row counts for every major selection step before adding graph or anomaly layers.